# 🎯 EduTutor — Notebook 2: Unsloth Fine-Tuning Pipeline

**Project:** EduTutor-Unsloth — A Neurodiversity-Affirming AI Tutor  
**Competition:** [Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon)  
**Tracks:** Future of Education + Unsloth Tech Track  
**Model:** Gemma 4 E4B (4B params, 128K context, multimodal, Apache 2.0)  

---

## Pipeline

```
Base Gemma 4 E4B
       │
       ▼
┌─────────────────┐
│  Stage 1: SFT   │  QLoRA + Unsloth → Teach the pedagogical persona
│  (Supervised)    │  Dataset: sft_train.jsonl from Notebook 1
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Stage 2: DPO   │  Direct Preference Optimization → Align away from
│  (Alignment)    │  answer-giving toward productive struggle
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Stage 3: Export │  Merge adapters → Export GGUF for local deployment
└─────────────────┘
```

### Hardware Requirements
- **GPU:** T4 (16GB) on Kaggle/Colab — sufficient for E4B QLoRA (~17GB peak with gradient checkpointing)
- **RAM:** 12GB+
- **Disk:** ~10GB for model weights + adapters

## 1. Environment Setup

In [ ]:
%%capture
# Install Unsloth — always use latest for Gemma 4 compatibility
!pip install -U unsloth
!pip install -q jsonlines datasets trl transformers accelerate bitsandbytes

In [ ]:
# --- Castalia/EduTutor Colab + Drive Setup ---
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path.cwd()

# Persist HuggingFace/Unsloth downloads and EduTutor outputs on Drive in Colab.
CACHE_DIR = DRIVE_ROOT / "models"
EDUTUTOR_ROOT = DRIVE_ROOT / "castalia-hackathons" / "edututor"
DATA_DIR = EDUTUTOR_ROOT / "data"
MODEL_DIR = EDUTUTOR_ROOT / "models"
EVAL_DIR = EDUTUTOR_ROOT / "eval"
SESSION_DIR = EDUTUTOR_ROOT / "sessions"

for path in [CACHE_DIR, DATA_DIR, MODEL_DIR, EVAL_DIR, SESSION_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["EDUTUTOR_CACHE_DIR"] = str(CACHE_DIR)

repo_candidates = [
    Path.cwd(),
    Path("/content/castalia"),
    DRIVE_ROOT / "castalia",
]
module_candidates = []
for candidate in repo_candidates:
    module_candidates.extend([
        candidate,
        candidate / "hackathons" / "gemma-4-good-hackathon" / "EduTutor",
    ])

module_dir = next((path for path in module_candidates if (path / "local_model.py").exists()), None)

if module_dir is None and IN_COLAB:
    repo_dir = Path("/content/castalia")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/cyruslayo/castalia.git", str(repo_dir)],
            check=True,
        )
    module_dir = repo_dir / "hackathons" / "gemma-4-good-hackathon" / "EduTutor"

if module_dir is None or not (module_dir / "local_model.py").exists():
    raise FileNotFoundError(
        "Could not find local_model.py. Run from the EduTutor folder, from the castalia repo root, "
        "or use the Colab bootstrap to clone castalia."
    )

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

print(f"EduTutor module path: {module_dir}")
print(f"Model cache: {CACHE_DIR}")
print(f"EduTutor outputs: {EDUTUTOR_ROOT}")

In [ ]:
import torch
import json
import jsonlines
from pathlib import Path
from datasets import Dataset

# Import the shared system prompt (single source of truth)
from local_model import EDUTUTOR_SYSTEM_PROMPT

# Verify GPU
assert torch.cuda.is_available(), "GPU required — enable T4 in Kaggle settings"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.version.cuda}")

## 2. Load Gemma 4 E4B with Unsloth

Unsloth patches the model for 2x faster training and 60% less VRAM. We load in 4-bit quantization (QLoRA) to fit on a T4.

In [ ]:
from unsloth import FastLanguageModel

# ---------- Model Configuration ----------
MODEL_NAME = "google/gemma-4-e4b"  # Gemma 4 Effective 4B
MAX_SEQ_LENGTH = 4096              # conversation context window
LOAD_IN_4BIT = True                # QLoRA quantization
DTYPE = None                       # auto-detect (float16 for T4)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=DTYPE,
    cache_dir=str(CACHE_DIR),
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Parameters: {model.num_parameters():,}")
print(f"   Quantization: {'4-bit QLoRA' if LOAD_IN_4BIT else 'Full precision'}")

## 3. Configure LoRA Adapters

We add LoRA adapters to the attention and MLP layers. Configuration choices:
- **r=32:** Higher rank for richer pedagogical personality capture
- **alpha=64:** Standard 2× rank scaling
- **Target modules:** All attention projections + gate/up/down MLPs for maximum expressiveness

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                          # LoRA rank — higher for complex persona
    lora_alpha=64,                 # scaling factor (2× rank)
    lora_dropout=0.05,             # slight regularization
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention
        "gate_proj", "up_proj", "down_proj",      # MLP
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=42,
)

# Print trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA adapters attached")
print(f"   Trainable params: {trainable:,} ({100*trainable/total:.2f}%)")
print(f"   Total params: {total:,}")

## 4. Load and Format SFT Dataset

We load the synthetic data from Notebook 1 and format it using the Gemma 4 chat template.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)


def load_sft_data(path: Path) -> Dataset:
    """Load SFT JSONL and format for the Gemma 4 chat template."""
    examples = []
    with jsonlines.open(path) as reader:
        for item in reader:
            messages = item["messages"]
            # Apply the Gemma 4 chat template
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            examples.append({"text": text})
    
    return Dataset.from_list(examples)


sft_train = load_sft_data(DATA_DIR / "sft_train.jsonl")
sft_val = load_sft_data(DATA_DIR / "sft_validation.jsonl")

print(f"✅ SFT data loaded")
print(f"   Train: {len(sft_train)} examples")
print(f"   Validation: {len(sft_val)} examples")
print(f"\n📝 Sample (first 500 chars):")
print(sft_train[0]["text"][:500])

## 5. Stage 1: Supervised Fine-Tuning (SFT)

This stage teaches the model the **EduTutor persona**: communication style, scaffolding patterns, emotional co-regulation, and profile-specific strategies.

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    # --- Output ---
    output_dir=str(MODEL_DIR / "edututor-sft-checkpoints"),
    
    # --- Training ---
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,      # effective batch size = 8
    warmup_ratio=0.1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    
    # --- Precision ---
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    # --- Evaluation ---
    eval_strategy="steps",
    eval_steps=50,
    
    # --- Saving ---
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    
    # --- Logging ---
    logging_steps=10,
    report_to="none",
    
    # --- Sequence ---
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,                       # pack short sequences for throughput
    dataset_text_field="text",
    
    # --- Seed ---
    seed=42,
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_train,
    eval_dataset=sft_val,
    args=sft_config,
)

print("✅ SFT Trainer configured")
print(f"   Epochs: {sft_config.num_train_epochs}")
print(f"   Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"   Learning rate: {sft_config.learning_rate}")

In [ ]:
# ---------- Train SFT ----------
print("🚀 Starting SFT training...")
sft_result = sft_trainer.train()

print(f"\n✅ SFT Training complete!")
print(f"   Total steps: {sft_result.global_step}")
print(f"   Final train loss: {sft_result.training_loss:.4f}")

# Save the SFT adapter
SFT_ADAPTER_DIR = MODEL_DIR / "edututor-sft-adapter"
model.save_pretrained(str(SFT_ADAPTER_DIR))
tokenizer.save_pretrained(str(SFT_ADAPTER_DIR))
print(f"   Adapter saved to: {SFT_ADAPTER_DIR}")

### 5.0.1 SFT Training Curves

Visualize the SFT training dynamics: loss convergence and learning rate schedule.

In [ ]:
import matplotlib.pyplot as plt

# ---------- Extract metrics from SFT trainer ----------
sft_log = sft_trainer.state.log_history

sft_train_steps = [e["step"] for e in sft_log if "loss" in e]
sft_train_loss  = [e["loss"] for e in sft_log if "loss" in e]
sft_eval_steps  = [e["step"] for e in sft_log if "eval_loss" in e]
sft_eval_loss   = [e["eval_loss"] for e in sft_log if "eval_loss" in e]
sft_lr_steps    = [e["step"] for e in sft_log if "learning_rate" in e]
sft_lr_values   = [e["learning_rate"] for e in sft_log if "learning_rate" in e]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Loss curves
ax1.plot(sft_train_steps, sft_train_loss, label="Train Loss", color="#2196F3", alpha=0.8)
if sft_eval_steps:
    ax1.plot(sft_eval_steps, sft_eval_loss, label="Eval Loss", color="#FF5722", marker="o",
             markersize=4, alpha=0.8)
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.set_title("SFT Loss Curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Panel 2: Learning rate schedule
ax2.plot(sft_lr_steps, sft_lr_values, color="#4CAF50", linewidth=2)
ax2.set_xlabel("Step")
ax2.set_ylabel("Learning Rate")
ax2.set_title("SFT Learning Rate Schedule (Cosine)")
ax2.grid(True, alpha=0.3)
ax2.ticklabel_format(axis="y", style="scientific", scilimits=(-4, -4))

fig.suptitle("SFT Training Dynamics", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(MODEL_DIR / "training_curves_sft.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------- Convergence Summary ----------
if sft_train_loss:
    init_loss = sft_train_loss[0]
    final_loss = sft_train_loss[-1]
    reduction = (1 - final_loss / init_loss) * 100
    print(f"\n📊 SFT Convergence Summary")
    print(f"   Initial train loss: {init_loss:.4f}")
    print(f"   Final train loss:   {final_loss:.4f}")
    print(f"   Reduction:          {reduction:.1f}%")
    if sft_eval_loss:
        print(f"   Final eval loss:    {sft_eval_loss[-1]:.4f}")
    print(f"   📈 Saved to: {MODEL_DIR / 'training_curves_sft.png'}")

### 5.1 Quick Sanity Check: SFT Model Response

Before moving to DPO, let's verify the SFT model has learned the basic persona.

In [ ]:
FastLanguageModel.for_inference(model)  # switch to inference mode

# Use the shared system prompt from local_model.py (single source of truth)
test_messages = [
    {"role": "system", "content": EDUTUTOR_SYSTEM_PROMPT},
    {"role": "user", "content": "I HATE fractions! I can't do this and I want to quit! I'm so STUPID!"}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=300,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print("🧒 Student: I HATE fractions! I can't do this and I want to quit! I'm so STUPID!")
print(f"\n🤖 EduTutor (SFT):")
print(response)
print("\n--- Sanity Checks ---")
print(f"  Contains empathy/validation: {'✅' if any(w in response.lower() for w in ['feel', 'hear', 'okay', 'understand', 'hard']) else '❌'}")
print(f"  Avoids giving answer: {'✅' if 'the answer is' not in response.lower() else '❌'}")
print(f"  Short sentences: {'✅' if len(response.split('.')[0].split()) < 20 else '⚠️  First sentence may be too long'}")

## 6. Stage 2: DPO Alignment

DPO further refines the model by training it to **prefer** scaffolding responses over answer-giving responses. This is the key alignment step that prevents the model from reverting to generic "helpful assistant" behavior.

In [ ]:
FastLanguageModel.for_training(model)  # switch back to training mode

# Ensure pad token is set for proper DPO collation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"✅ Set pad_token = eos_token ({tokenizer.eos_token})")


def load_dpo_data(path: Path) -> Dataset:
    """Load DPO JSONL into HF Dataset format expected by TRL's DPOTrainer."""
    examples = []
    with jsonlines.open(path) as reader:
        for item in reader:
            # Format prompt messages
            prompt_text = tokenizer.apply_chat_template(
                item["prompt"],
                tokenize=False,
                add_generation_prompt=True,
            )
            examples.append({
                "prompt": prompt_text,
                "chosen": item["chosen"][0]["content"],
                "rejected": item["rejected"][0]["content"],
            })
    
    return Dataset.from_list(examples)


dpo_train = load_dpo_data(DATA_DIR / "dpo_train.jsonl")
dpo_val = load_dpo_data(DATA_DIR / "dpo_validation.jsonl")

print(f"✅ DPO data loaded")
print(f"   Train: {len(dpo_train)} preference pairs")
print(f"   Validation: {len(dpo_val)} preference pairs")

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir=str(MODEL_DIR / "edututor-dpo-checkpoints"),
    
    # --- Training ---
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,       # effective batch size = 8
    learning_rate=5e-5,                  # lower LR for alignment
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    
    # --- DPO Specific ---
    beta=0.1,                            # KL penalty coefficient
    loss_type="sigmoid",                 # standard DPO loss
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=1024,
    
    # --- Precision ---
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    # --- Evaluation ---
    eval_strategy="steps",
    eval_steps=25,
    
    # --- Saving ---
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    
    # --- Logging ---
    logging_steps=5,
    report_to="none",
    
    seed=42,
)

dpo_trainer = DPOTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dpo_train,
    eval_dataset=dpo_val,
    args=dpo_config,
)

print("✅ DPO Trainer configured")
print(f"   Beta (KL penalty): {dpo_config.beta}")
print(f"   Learning rate: {dpo_config.learning_rate}")

In [ ]:
# ---------- Train DPO ----------
print("🚀 Starting DPO alignment...")
dpo_result = dpo_trainer.train()

print(f"\n✅ DPO Alignment complete!")
print(f"   Total steps: {dpo_result.global_step}")
print(f"   Final train loss: {dpo_result.training_loss:.4f}")

# Save the DPO-aligned adapter
DPO_ADAPTER_DIR = MODEL_DIR / "edututor-dpo-adapter"
model.save_pretrained(str(DPO_ADAPTER_DIR))
tokenizer.save_pretrained(str(DPO_ADAPTER_DIR))
print(f"   Adapter saved to: {DPO_ADAPTER_DIR}")

### 6.1 DPO Training Curves

Visualize the DPO alignment training dynamics.

In [ ]:
# ---------- Extract metrics from DPO trainer ----------
dpo_log = dpo_trainer.state.log_history

dpo_train_steps = [e["step"] for e in dpo_log if "loss" in e]
dpo_train_loss  = [e["loss"] for e in dpo_log if "loss" in e]
dpo_eval_steps  = [e["step"] for e in dpo_log if "eval_loss" in e]
dpo_eval_loss   = [e["eval_loss"] for e in dpo_log if "eval_loss" in e]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(dpo_train_steps, dpo_train_loss, label="Train Loss", color="#9C27B0", alpha=0.8)
if dpo_eval_steps:
    ax.plot(dpo_eval_steps, dpo_eval_loss, label="Eval Loss", color="#FF9800", marker="o",
            markersize=4, alpha=0.8)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("DPO Alignment Training Curves", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(MODEL_DIR / "training_curves_dpo.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------- Convergence Summary ----------
if dpo_train_loss:
    init_loss = dpo_train_loss[0]
    final_loss = dpo_train_loss[-1]
    reduction = (1 - final_loss / init_loss) * 100
    print(f"\n📊 DPO Convergence Summary")
    print(f"   Initial train loss: {init_loss:.4f}")
    print(f"   Final train loss:   {final_loss:.4f}")
    print(f"   Reduction:          {reduction:.1f}%")
    if dpo_eval_loss:
        print(f"   Final eval loss:    {dpo_eval_loss[-1]:.4f}")
    print(f"   📈 Saved to: {MODEL_DIR / 'training_curves_dpo.png'}")

## 7. Stage 3: Export

We merge the LoRA adapters into the base model and export to multiple formats:
- **GGUF (Q4_K_M):** For llama.cpp / Ollama local deployment
- **Safetensors:** For HuggingFace Hub / vLLM serving

In [ ]:
# ---------- Merge and Export ----------
MERGED_DIR = MODEL_DIR / "edututor-merged"
GGUF_DIR = MODEL_DIR / "edututor-gguf"

# Export to GGUF for local deployment (Ollama/llama.cpp)
print("📦 Exporting to GGUF (Q4_K_M quantization)...")
model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method="q4_k_m",  # good quality/size balance
)
print(f"   ✅ GGUF saved to: {GGUF_DIR}")

# Also save merged safetensors for HF Hub
print("\n📦 Exporting merged safetensors...")
model.save_pretrained_merged(
    str(MERGED_DIR),
    tokenizer,
    save_method="merged_16bit",
)
print(f"   ✅ Merged model saved to: {MERGED_DIR}")

# List exported files
import os
print(f"\n📁 GGUF files:")
for f in os.listdir(GGUF_DIR):
    size_mb = os.path.getsize(GGUF_DIR / f) / 1e6
    print(f"   {f} ({size_mb:.1f} MB)")

### 7.1 Export Verification

Verify that all exported files are present, valid, and correctly formatted.

In [ ]:
import os
import struct

print("=" * 60)
print("🔍 MODEL EXPORT VERIFICATION")
print("=" * 60)

all_ok = True

# ---------- GGUF Verification ----------
print(f"\n📦 GGUF Directory: {GGUF_DIR}")
if GGUF_DIR.is_dir():
    gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith(".gguf")]
    if gguf_files:
        for fname in gguf_files:
            fpath = GGUF_DIR / fname
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            # Verify GGUF magic bytes
            with open(fpath, "rb") as f:
                magic = f.read(4)
            is_valid = magic == b"GGUF"
            icon = "✅" if is_valid else "❌"
            print(f"   {icon} {fname}: {size_mb:.1f} MB"
                  f" | Magic: {magic!r} {'(valid)' if is_valid else '(INVALID!)'}")
            if not is_valid:
                all_ok = False
    else:
        print("   ❌ No .gguf files found!")
        all_ok = False

    # Check for config files in GGUF dir
    for cfg in ["config.json", "tokenizer.json"]:
        cfg_path = GGUF_DIR / cfg
        if os.path.isfile(cfg_path):
            print(f"   ✅ {cfg} present")
        else:
            print(f"   ⚠️  {cfg} not found (may be normal for GGUF-only export)")
else:
    print(f"   ❌ Directory not found: {GGUF_DIR}")
    all_ok = False

# ---------- Safetensors Verification ----------
print(f"\n📦 Merged Safetensors Directory: {MERGED_DIR}")
if MERGED_DIR.is_dir():
    st_files = [f for f in os.listdir(MERGED_DIR) if f.endswith(".safetensors")]
    if st_files:
        total_size = 0
        for fname in st_files:
            fpath = MERGED_DIR / fname
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            total_size += size_mb
            print(f"   ✅ {fname}: {size_mb:.1f} MB")
        print(f"   📊 Total safetensors: {total_size:.1f} MB ({len(st_files)} shards)")
    else:
        print("   ❌ No .safetensors files found!")
        all_ok = False

    # Check for config files
    for cfg in ["config.json", "tokenizer.json"]:
        cfg_path = MERGED_DIR / cfg
        if os.path.isfile(cfg_path):
            size_kb = os.path.getsize(cfg_path) / 1024
            print(f"   ✅ {cfg} present ({size_kb:.1f} KB)")
        else:
            print(f"   ❌ {cfg} MISSING!")
            all_ok = False
else:
    print(f"   ❌ Directory not found: {MERGED_DIR}")
    all_ok = False

# ---------- Summary ----------
print(f"\n{'=' * 60}")
if all_ok:
    print("✅ ALL EXPORT CHECKS PASSED — model artifacts are ready for deployment")
else:
    print("⚠️  SOME CHECKS FAILED — review the issues above")
print(f"{'=' * 60}")

## 8. Final Verification: Expanded Sanity Checks

We test the DPO-aligned model against **4 targeted scenarios** covering all emotional zones and neurodivergence profiles. Each scenario is scored on:
- **Expected pattern presence** (70%): Does the response contain empathetic, pedagogically appropriate language?
- **Forbidden pattern absence** (30%): Does the response avoid answer-giving and dismissive language?

A quality score ≥ 0.6 is considered passing.

In [ ]:
FastLanguageModel.for_inference(model)

# ---------- 4 Targeted Test Scenarios ----------
test_scenarios = [
    {
        "label": "🔴 RED Zone — Crisis (Dyscalculia)",
        "messages": [
            {"role": "system", "content": EDUTUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": "I CAN'T DO MATH! I'm the WORST! Everyone finishes before me! I want to QUIT!"},
        ],
        "expected_patterns": ["hear", "understand", "hard", "okay", "feel"],
        "forbidden_patterns": ["the answer is", "just multiply", "it's easy", "simple"],
    },
    {
        "label": "🟢 GREEN Zone — Ready to Learn (ADHD)",
        "messages": [
            {"role": "system", "content": EDUTUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": "I'm ready! Let's do fractions! But can we make it quick? I get bored easily."},
        ],
        "expected_patterns": ["great", "let's", "start", "one", "step"],
        "forbidden_patterns": ["just focus", "pay attention", "sit still", "try harder"],
    },
    {
        "label": "🟡 YELLOW Zone — Frustrated (Autism)",
        "messages": [
            {"role": "system", "content": EDUTUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": "This doesn't make sense. You said we'd do multiplication but now it's division. I don't like when things change. I want to stop."},
        ],
        "expected_patterns": ["understand", "change", "okay", "plan", "know"],
        "forbidden_patterns": ["the answer is", "it's easy", "just try", "move on"],
    },
    {
        "label": "🔵 BLUE Zone — Withdrawn (Dyslexia)",
        "messages": [
            {"role": "system", "content": EDUTUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": "i dont want to read anymore. the words are all mixed up. i cant do it."},
        ],
        "expected_patterns": ["hear", "okay", "hard", "together", "help"],
        "forbidden_patterns": ["read this", "sound it out", "it's easy", "just try"],
    },
]

# ---------- Run & Score Each Scenario ----------
results = []
print("=" * 70)
print("🧪 EXPANDED SANITY CHECK — 4 Neurodiversity Scenarios")
print("=" * 70)

for scenario in test_scenarios:
    inputs = tokenizer.apply_chat_template(
        scenario["messages"],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    resp_lower = response.lower()

    # Score: expected pattern presence
    expected = scenario["expected_patterns"]
    found_expected = [p for p in expected if p in resp_lower]
    expected_ratio = len(found_expected) / len(expected) if expected else 1.0

    # Score: forbidden pattern absence
    forbidden = scenario["forbidden_patterns"]
    found_forbidden = [p for p in forbidden if p in resp_lower]
    no_forbidden = len(found_forbidden) == 0

    quality_score = 0.7 * expected_ratio + 0.3 * (1.0 if no_forbidden else 0.0)
    passed = quality_score >= 0.6

    results.append({
        "label": scenario["label"],
        "score": quality_score,
        "passed": passed,
        "response": response,
        "found_expected": found_expected,
        "found_forbidden": found_forbidden,
    })

    # Print formatted results
    print(f"\n{'━' * 60}")
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"📋 {scenario['label']}  [{status}]  Score: {quality_score:.2f}")
    print(f"{'━' * 60}")
    print(f"🧒 Student: {scenario['messages'][-1]['content']}")
    print(f"\n🤖 EduTutor (SFT+DPO):")
    print(response)
    print(f"\n   Expected patterns found: {len(found_expected)}/{len(expected)}"
          f"  {found_expected}")
    if found_forbidden:
        print(f"   ⚠️  Forbidden patterns found: {found_forbidden}")
    else:
        print(f"   ✅ No forbidden patterns found")

# ---------- Overall Summary ----------
total = len(results)
passed_count = sum(1 for r in results if r["passed"])
avg_score = sum(r["score"] for r in results) / total if total else 0

print(f"\n{'=' * 70}")
print(f"📊 OVERALL SUMMARY")
print(f"{'=' * 70}")
print(f"   Scenarios passed: {passed_count}/{total}")
print(f"   Average quality score: {avg_score:.2f}")
for r in results:
    icon = "✅" if r["passed"] else "❌"
    print(f"   {icon} {r['label']}: {r['score']:.2f}")

overall_status = "✅ ALL CHECKS PASSED" if passed_count == total else "⚠️  SOME CHECKS FAILED"
print(f"\n   {overall_status}")

In [ ]:
# ---------- Generate HuggingFace Model Card ----------
import os

# Gather training metrics for the model card
sft_final_loss = sft_result.training_loss if hasattr(sft_result, "training_loss") else "N/A"
dpo_final_loss = dpo_result.training_loss if hasattr(dpo_result, "training_loss") else "N/A"

model_card = f'''---
license: apache-2.0
tags:
  - education
  - neurodiversity
  - fine-tuned
  - Gemma4
  - accessibility
  - special-education
base_model: google/gemma-4-e4b
pipeline_tag: text-generation
language:
  - en
---

# 🎯 EduTutor — Neurodiversity-Affirming AI Tutor

A fine-tuned [Gemma 4 E4B](https://huggingface.co/google/gemma-4-e4b) model specialized for
tutoring neurodivergent children aged 8–14. Built for the
[Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon).

## Model Details

| Field | Value |
|---|---|
| **Base model** | `google/gemma-4-e4b` (4B params) |
| **Fine-tuning** | QLoRA (r=32, α=64) → SFT → DPO |
| **Training data** | Synthetic neurodiversity tutoring conversations |
| **Quantization** | 4-bit (QLoRA) during training; exported as Q4_K_M GGUF + FP16 safetensors |

## Training Details

### Stage 1: Supervised Fine-Tuning (SFT)
- **Epochs:** 3
- **Effective batch size:** 8 (2 × 4 gradient accumulation)
- **Learning rate:** 2e-4 (cosine schedule, 10% warmup)
- **Sequence packing:** enabled
- **Final train loss:** {sft_final_loss:.4f}

### Stage 2: Direct Preference Optimization (DPO)
- **Epochs:** 2
- **Effective batch size:** 8 (1 × 8 gradient accumulation)
- **Learning rate:** 5e-5 (cosine schedule, 10% warmup)
- **Beta (KL penalty):** 0.1
- **Loss type:** sigmoid
- **Final train loss:** {dpo_final_loss:.4f}

### LoRA Configuration
- **Rank (r):** 32
- **Alpha:** 64
- **Dropout:** 0.05
- **Target modules:** q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj

## Intended Use

EduTutor is designed to help **neurodivergent children aged 8–14** with:
- **ADHD:** Micro-chunked tasks, novelty, immediate celebration of wins
- **Autism:** Explicit structure, transition warnings, interest connections
- **Dyslexia:** Minimal text, Orton-Gillingham phonics, no forced reading aloud
- **Dyscalculia:** CRA progression (Concrete → Representational → Abstract), no timed tasks

### Pedagogical Strategies
- **Productive struggle:** Never gives answers directly; scaffolds toward discovery
- **Emotional co-regulation:** Monitors emotional zones (Green/Yellow/Red/Blue)
- **Cognitive load management:** Short sentences, one idea at a time
- **Strengths-based:** Presumes competence, celebrates what children CAN do

## Limitations

- **Synthetic training data:** All conversations are LLM-generated, not from real tutoring sessions
- **Text-only:** Does not process images, audio, or video input
- **English-only:** Not trained on other languages
- **Not a replacement for professional support:** Should complement, not replace, trained special educators
- **Safety:** Not evaluated for all possible harmful outputs; adult supervision recommended

## Citation

```bibtex
@misc{{edututor2025,
  title={{EduTutor: A Neurodiversity-Affirming AI Tutor Fine-Tuned on Gemma 4}},
  author={{EduTutor Team}},
  year={{2025}},
  howpublished={{Gemma 4 Good Hackathon — Kaggle}},
  url={{https://www.kaggle.com/competitions/gemma-4-good-hackathon}},
  note={{Fine-tuned with QLoRA SFT + DPO using Unsloth}}
}}
```
'''

# Write to merged model export directory
card_path = MERGED_DIR / "README.md"
MERGED_DIR.mkdir(parents=True, exist_ok=True)
with open(card_path, "w", encoding="utf-8") as f:
    f.write(model_card)

print(f"✅ Model card written to: {card_path}")
print(f"   Size: {len(model_card):,} characters")
print(f"   Tags: education, neurodiversity, fine-tuned, Gemma4, accessibility, special-education")

## Summary

| Stage | What It Did | Output |
|---|---|---|
| **SFT** | Taught the model the EduTutor persona, scaffolding patterns, and profile-specific strategies | LoRA adapter in `edututor-sft-adapter/` |
| **DPO** | Aligned the model to prefer productive struggle over answer-giving | LoRA adapter in `edututor-dpo-adapter/` |
| **Export** | Merged adapters and exported for deployment | GGUF in `edututor-gguf/`, safetensors in `edututor-merged/` |

### Next Step

→ **Notebook 3: `03_evaluation_harness.ipynb`** — Formally evaluate the fine-tuned model against base Gemma 4 using custom pedagogical rubrics and LLM-as-Judge.